# Babaorum lecteur — R3 « Ancre » : encodeur dots gelé + CTC + décodeur, décodage joint ancré

Plan : `docs/READER_PLAN.txt` (onglet lecteur, Claude w2:p9). **Lancer seulement avec l'accord de Marcel.**
Phases : F (caractéristiques L = 32 en cache local), T (entraînement multitâche, tête CTC partie de A1),
E (dev : réglage sur la moitié tune, mesure sur la moitié eval), I (invention sur images perturbées).
Durée visée ≈ 3 h sur T4. Sorties : `MyDrive/babaorum/reader_r3/`. Aucune ligne de test n'est lue.

In [ ]:
import os, sys, json, gzip, glob, hashlib, shutil, subprocess, time, zipfile, math, random
KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if KAGGLE:   # entrées = jeux Kaggle (axel-e8-presse, sortie de axel-e17-vitctc), aucun consentement Drive
    D, OUT, TMP = '/kaggle/input', '/kaggle/working/reader_r3', '/kaggle/tmp'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/babaorum/in'
    OUT = '/content/drive/MyDrive/babaorum/reader_r3'
    TMP = '/content/tmp'
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
T_START = time.time()
PHASES = ('F', 'T', 'E', 'I')
L = 32
REPO, REV = 'dots-studio/dots.mocr', 'e539fbb52280393adc081b289ec597430a0f9031'   # révision d'E3/E17
BUDGET_H = 3.5            # plus de nouvelle époque au-delà ; E et I se lancent quand même
EPOCHS, LAMBDA_CTC = 12, 0.3
SMOKE = True             # True : essai court (compte gratuit, < 1 h) : 2 000 lignes train, 300 dev, 1 époque
if SMOKE:
    EPOCHS, BUDGET_H = 1, 0.8
    OUT = OUT + '_smoke'; os.makedirs(OUT, exist_ok=True)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
%%bash
pip install -q safetensors huggingface_hub rapidfuzz 2>&1 | tail -2
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## Poids : seulement `vision_tower.*`

In [ ]:
import torch
from safetensors import safe_open
from safetensors.torch import save_file
VIS = f'{TMP}/vision.safetensors'
have = sorted(glob.glob(f'{D}/**/vision.safetensors', recursive=True))
if have:
    VIS = have[0]
elif not os.path.exists(VIS):
    from huggingface_hub import hf_hub_download
    idx = json.load(open(hf_hub_download(REPO, 'model.safetensors.index.json', revision=REV, cache_dir=f'{TMP}/hf')))
    vis = {}
    for shard in sorted({v for k, v in idx['weight_map'].items() if k.startswith('vision_tower.')}):
        p = hf_hub_download(REPO, shard, revision=REV, cache_dir=f'{TMP}/hf')
        with safe_open(p, framework='pt') as fh:
            for k in fh.keys():
                if k.startswith('vision_tower.'):
                    vis[k] = fh.get_tensor(k)
        os.remove(os.path.realpath(p))
    save_file(vis, VIS)
    shutil.rmtree(f'{TMP}/hf', ignore_errors=True)
with safe_open(VIS, framework='pt') as fh:
    n = sum(math.prod(fh.get_slice(k).get_shape()) for k in fh.keys())
print(VIS, os.path.getsize(VIS) >> 20, 'Mo', n, 'paramètres')   # attendu 1 262 091 264

## Recadrages (comme E8 : archives publiques, empreinte MD5 contrôlée)

In [ ]:
def find(pattern):
    return sorted(glob.glob(f'{D}/**/{pattern}', recursive=True))
items_path = (find('e8_items.jsonl.gz') or find('e8_items.jsonl'))[0]
op = gzip.open if items_path.endswith('.gz') else open
with op(items_path, 'rt', encoding='utf-8') as fh:
    items = [json.loads(l) for l in fh if l.strip()]
sys.path.insert(0, os.path.dirname(find('crops.py')[0]))
from crops import crop_line
need = {}
for it in items:
    kind, name = it['image'].split('/', 1)
    need.setdefault(kind, set()).add(name)
print(len(items), 'lignes', {k: len(v) for k, v in need.items()})

NEWSEYE = 'https://zenodo.org/records/4293602/files/ATR_TrainingSet_BnF_Newseye_M2+.zip?download=1'
BNL = 'https://data.bnl.lu/wp-content/uploads/2021/07/bnl-ground-truth-newspapers-before-1878.zip'
def fetch(url, path):
    if not os.path.exists(path):
        subprocess.run(['wget', '-q', '-O', path, url], check=True)
def extract(zf, wanted, dest):
    os.makedirs(dest, exist_ok=True)
    for info in zf.infolist():
        base = os.path.basename(info.filename)
        if base in wanted and not os.path.exists(f'{dest}/{base}'):
            with zf.open(info) as s, open(f'{dest}/{base}', 'wb') as d:
                shutil.copyfileobj(s, d)
IMG = f'{TMP}/img'
if not all(os.path.exists(f'{IMG}/newseye/{n}') for n in need['newseye']):
    fetch(NEWSEYE, f'{TMP}/newseye.zip')
    with zipfile.ZipFile(f'{TMP}/newseye.zip') as zf:
        extract(zf, need['newseye'], f'{IMG}/newseye')
    os.remove(f'{TMP}/newseye.zip')
if not all(os.path.exists(f'{IMG}/bnl/{n}') for n in need['bnl']):
    fetch(BNL, f'{TMP}/bnl.zip')
    with zipfile.ZipFile(f'{TMP}/bnl.zip') as zf:
        inner = next(i for i in zf.infolist() if i.filename.endswith('/antiqua.zip'))
        with zf.open(inner) as s, open(f'{TMP}/antiqua.zip', 'wb') as d:
            shutil.copyfileobj(s, d)
    os.remove(f'{TMP}/bnl.zip')
    with zipfile.ZipFile(f'{TMP}/antiqua.zip') as zf:
        extract(zf, need['bnl'], f'{IMG}/bnl')
    os.remove(f'{TMP}/antiqua.zip')

In [ ]:
# Recadrages gardés en mémoire (niveaux de gris, ~2 Go) ; empreinte MD5 comparée à celle du Mac.
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
by_image = {}
for it in items:
    by_image.setdefault(it['image'], []).append(it)
CROPS, mism = {}, []
t0 = time.time()
for img, group in by_image.items():
    with Image.open(f'{IMG}/{img}') as raw_img:
        page = raw_img.convert('L')
    for it in group:
        im = page if img.startswith('bnl/') else crop_line(page, tuple(it['box']), it['polygon'], pad=4).image
        if hashlib.md5(im.tobytes()).hexdigest() != it['crop_md5']:
            mism.append(it['stem'])
        CROPS[it['stem']] = im.copy()
TRAIN = [it for it in items if it['split'] == 'train']
DEV = [it for it in items if it['split'] == 'dev']
print(len(TRAIN), len(DEV), 'md5 différents :', len(mism), round(time.time() - t0), 's')

## Encodeur, tête, lecture, CER

Recopiés de `src/axel/reader/` par le générateur (source unique, C3) : ne pas modifier ici.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, unicodedata
# --- axel/reader/vit.py, recopié par le générateur : ne pas modifier ici ---
import math
from collections.abc import Sequence
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn

MEAN = (0.48145466, 0.4578275, 0.40821073)
STD = (0.26862954, 0.26130258, 0.27577711)
PATCH = 14

#: largeur, têtes, SwiGLU, blocs, RoPE, préfixe des poids (dépôt HF épinglé)
SPECS: dict[str, dict[str, Any]] = {
    "dots_mocr": {
        "dim": 1536,
        "heads": 12,
        "ffn": 4224,
        "depth": 42,
        "rope": "concat",
        "prefix": "vision_tower.",
        "repo": "dots-studio/dots.mocr",
        "rev": "e539fbb52280393adc081b289ec597430a0f9031",
    },
    "monkeyocrv2_s": {
        "dim": 384,
        "heads": 6,
        "ffn": 1536,
        "depth": 12,
        "rope": "interleave",
        "prefix": "",
        "repo": "zenosai/MonkeyOCRv2-S",
        "rev": "8efa4f856af5d38030d83565c8a79b36e5454ce7",
    },
    "monkeyocrv2_b": {
        "dim": 768,
        "heads": 12,
        "ffn": 3072,
        "depth": 12,
        "rope": "interleave",
        "prefix": "",
        "repo": "zenosai/MonkeyOCRv2-B",
        "rev": "85170aa4a1d87a2206920a3e09f323474f91dc33",
    },
}


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return out.type_as(x) * self.weight


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


class Block(nn.Module):
    def __init__(self, dim: int, heads: int, ffn: int) -> None:
        super().__init__()
        self.heads = heads
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.fc1 = nn.Linear(dim, ffn, bias=False)
        self.fc2 = nn.Linear(ffn, dim, bias=False)
        self.fc3 = nn.Linear(dim, ffn, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        b, n, d = x.shape
        q, k, v = (
            self.qkv(self.norm1(x))
            .view(b, n, 3, self.heads, -1)
            .permute(2, 0, 3, 1, 4)
            .unbind(0)
        )
        qf, kf = q.float(), k.float()  # rotation en float32, comme les originaux
        q = (qf * cos + rotate_half(qf) * sin).to(v.dtype)
        k = (kf * cos + rotate_half(kf) * sin).to(v.dtype)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.proj(a.transpose(1, 2).reshape(b, n, d))
        h = self.norm2(x)
        y: torch.Tensor = x + self.fc2(F.silu(self.fc1(h)) * self.fc3(h))
        return y


def key_mask(
    widths: Sequence[int], h: int, w: int, device: torch.device
) -> torch.Tensor | None:
    """Masque booléen [b, 1, 1, h·w] des clés valides (ordre raster), ou
    ``None`` si toutes les images du lot ont la largeur ``w``."""
    wt = torch.as_tensor(list(widths), device=device)
    if bool((wt >= w).all()):
        return None
    col = torch.arange(w, device=device).repeat(h)  # colonne de chaque jeton
    return (col[None, :] < wt[:, None])[:, None, None, :]


class NaViT(nn.Module):
    """Les ``blocks`` premiers blocs d'un encodeur de ``SPECS``."""

    def __init__(
        self, dim: int, heads: int, ffn: int, blocks: int, rope: str = "concat"
    ) -> None:
        super().__init__()
        self.dim, self.rope_mode = dim, rope
        self.proj = nn.Conv2d(3, dim, PATCH, PATCH)
        self.pnorm = RMSNorm(dim)
        self.blocks = nn.ModuleList(Block(dim, heads, ffn) for _ in range(blocks))
        self.post = RMSNorm(dim)
        hd = dim // heads
        inv = 1.0 / (
            10000 ** (torch.arange(0, hd // 2, 2, dtype=torch.float) / (hd // 2))
        )
        self.register_buffer("inv_freq", inv, persistent=False)

    def rope(
        self, h: int, w: int, device: torch.device
    ) -> tuple[torch.Tensor, torch.Tensor]:
        hp = torch.arange(h, device=device).unsqueeze(1).expand(h, w).flatten()
        wp = torch.arange(w, device=device).unsqueeze(0).expand(h, w).flatten()
        inv = self.get_buffer("inv_freq")
        full = torch.outer(
            torch.arange(max(h, w), device=device, dtype=torch.float), inv
        )
        if self.rope_mode == "concat":
            emb = torch.cat([full[hp], full[wp]], dim=-1)
        else:
            emb = torch.stack([full[hp], full[wp]], dim=-1).flatten(1)
        emb = emb.repeat(1, 2)
        return emb.cos()[None, None], emb.sin()[None, None]

    def forward(
        self,
        img: torch.Tensor,
        taps: tuple[int, ...] = (),
        post: bool = True,
        widths: Sequence[int] | None = None,
    ) -> dict[int, torch.Tensor]:
        """img [b, 3, H, W] normalisée → {profondeur: [b, h, w, dim]} ; la
        dernière profondeur gardée passe par ``post_trunk_norm`` si ``post``.
        ``widths`` : colonnes de patchs valides de chaque image (masque M1) ;
        les colonnes au-delà sont du remplissage, leur sortie n'a pas de sens.
        """
        b, _, hh, ww = img.shape
        h, w = hh // PATCH, ww // PATCH
        x = self.proj(img.to(self.proj.weight.dtype)).flatten(2).transpose(1, 2)
        x = self.pnorm(x)
        cos, sin = self.rope(h, w, img.device)
        mask = None if widths is None else key_mask(widths, h, w, img.device)
        out: dict[int, torch.Tensor] = {}
        n = len(self.blocks)
        for i, blk in enumerate(self.blocks, 1):
            x = blk(x, cos, sin, mask)
            if i in taps and i != n:
                out[i] = x.view(b, h, w, self.dim)
        if n in taps or not taps:
            out[n] = (self.post(x) if post else x).view(b, h, w, self.dim)
        return out


def rename_map(prefix: str, blocks: int) -> dict[str, str]:
    ren = {
        prefix + "patch_embed.patchifier.proj.weight": "proj.weight",
        prefix + "patch_embed.patchifier.proj.bias": "proj.bias",
        prefix + "patch_embed.patchifier.norm.weight": "pnorm.weight",
        prefix + "post_trunk_norm.weight": "post.weight",
    }
    for i in range(blocks):
        p = f"{prefix}blocks.{i}."
        for a, b in [
            ("norm1", "norm1"),
            ("norm2", "norm2"),
            ("attn.qkv", "qkv"),
            ("attn.proj", "proj"),
            ("mlp.fc1", "fc1"),
            ("mlp.fc2", "fc2"),
            ("mlp.fc3", "fc3"),
        ]:
            ren[p + a + ".weight"] = f"blocks.{i}.{b}.weight"
    return ren


def load(
    name: str, path: str, blocks: int | None = None, dtype: torch.dtype = torch.float32
) -> NaViT:
    """Encodeur ``name`` depuis un fichier ``safetensors`` (ses seuls tenseurs)."""
    from safetensors import safe_open

    s = SPECS[name]
    n = blocks or s["depth"]
    m = NaViT(s["dim"], s["heads"], s["ffn"], n, s["rope"])
    sd = {}
    with safe_open(path, framework="pt") as fh:
        for k, v in rename_map(s["prefix"], n).items():
            sd[v] = fh.get_tensor(k)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    assert not unexpected and not [k for k in missing if k != "inv_freq"], (
        missing,
        unexpected,
    )
    return m.to(dtype).eval()


def prep(img: Image.Image, height: int = 56, max_w: int = 2800) -> torch.Tensor:
    """Ligne → [3, height, W] normalisé CLIP, W multiple de 28 (= E17, D-261)."""
    g = img.convert("L")
    w = min(max_w, max(28, round(g.width * height / max(1, g.height))))
    g = g.resize((w, height), Image.Resampling.BICUBIC)
    canvas = Image.new("L", (math.ceil(w / 28) * 28, height), 255)
    canvas.paste(g, (0, 0))
    a = torch.from_numpy(np.asarray(canvas, dtype=np.float32) / 255.0)
    return (
        a.unsqueeze(0).expand(3, -1, -1) - torch.tensor(MEAN).view(3, 1, 1)
    ) / torch.tensor(STD).view(3, 1, 1)


def qwen_patches(
    x: torch.Tensor, merge: int = 2
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """[3, H, W] → patchs dans l'ordre du processeur Qwen2-VL (blocs 2 × 2),
    ``grid_thw`` et la permutation qui ramène cet ordre à l'ordre raster."""
    c, hh, ww = x.shape
    h, w = hh // PATCH, ww // PATCH
    p = x.reshape(c, h // merge, merge, PATCH, w // merge, merge, PATCH)
    p = p.permute(1, 4, 2, 5, 0, 3, 6).reshape(h * w, c * PATCH * PATCH)
    idx = (
        torch.arange(h * w)
        .reshape(h // merge, merge, w // merge, merge)
        .permute(0, 2, 1, 3)
        .flatten()
    )
    return p, torch.tensor([[1, h, w]]), idx

# --- axel/reader/ctc.py, recopié par le générateur : ne pas modifier ici ---
from collections.abc import Callable, Sequence
from contextlib import AbstractContextManager, nullcontext

import torch
from torch import nn


class Head(nn.Module):
    """Tête d'E17 (D-264) : LayerNorm, Dropout 0,1, linéaire → ``hid``,
    BiLSTM 2 × ``hid``, linéaire vers le codec ; chaque colonne est répétée
    ``up`` fois. Mêmes noms de poids qu'avant C3 (``head_a1.pt`` se charge)."""

    def __init__(
        self, n_out: int, din: int = 1536, hid: int = 256, up: int = 2
    ) -> None:
        super().__init__()
        self.up = up
        self.inp = nn.Sequential(
            nn.LayerNorm(din), nn.Dropout(0.1), nn.Linear(din, hid)
        )
        self.rnn = nn.LSTM(
            hid, hid, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1
        )
        self.out = nn.Linear(2 * hid, n_out)

    def forward(
        self, x: torch.Tensor, lengths: Sequence[int] | torch.Tensor | None = None
    ) -> torch.Tensor:
        """x [b, T, din] → logits [b, T·up, n_out]. ``lengths`` : colonnes
        valides de chaque ligne ; le BiLSTM ne voit qu'elles (suites
        « empaquetées »). Sans ``lengths``, toute la suite est lue : réservé
        à une ligne seule, ou à la reproduction de l'ancien calcul."""
        z = self.inp(x).repeat_interleave(self.up, dim=1)
        if lengths is None:
            out: torch.Tensor = self.out(self.rnn(z)[0])
            return out
        n = torch.as_tensor(lengths, dtype=torch.long).cpu() * self.up
        packed = nn.utils.rnn.pack_padded_sequence(
            z, n, batch_first=True, enforce_sorted=False
        )
        y = nn.utils.rnn.pad_packed_sequence(
            self.rnn(packed)[0], batch_first=True, total_length=z.shape[1]
        )[0]
        logits: torch.Tensor = self.out(y)
        return logits


def greedy(lp: torch.Tensor, chars: Sequence[str]) -> str:
    """Lecture gloutonne CTC d'une ligne [T, n_out] (blanc = 0)."""
    out, prev = [], 0
    for b in lp.argmax(-1).tolist():
        if b != prev and b != 0:
            out.append(chars[b - 1])
        prev = b
    return "".join(out)


def white_value(mean: Sequence[float], std: Sequence[float]) -> torch.Tensor:
    """Pixel blanc normalisé [3, 1, 1] (remplissage, comme ``prep``)."""
    return ((1 - torch.tensor(mean)) / torch.tensor(std)).view(3, 1, 1)


def pad_images(
    xs: Sequence[torch.Tensor], white: torch.Tensor, patch: int = 14
) -> tuple[torch.Tensor, list[int]]:
    """[3, H, w_j] → lot [b, 3, H, max w] complété en blanc, et colonnes de
    patchs valides de chaque image."""
    c, hh = xs[0].shape[0], xs[0].shape[1]
    ww = max(x.shape[-1] for x in xs)
    xb = white.to(xs[0].dtype).expand(len(xs), c, hh, ww).clone()
    for j, x in enumerate(xs):
        xb[j, :, :, : x.shape[-1]] = x
    return xb, [x.shape[-1] // patch for x in xs]


def encode_lines(
    enc: nn.Module,
    xs: Sequence[torch.Tensor],
    depth: int,
    white: torch.Tensor,
    device: torch.device | str = "cpu",
    amp: Callable[[], AbstractContextManager[object]] | None = None,
    patch: int = 14,
) -> list[torch.Tensor]:
    """Caractéristiques [w_j, dim] (float32) de chaque ligne à la profondeur
    ``depth``, moyennées sur la hauteur ; encodeur masqué (``widths``)."""
    xb, widths = pad_images(xs, white, patch)
    with (amp or nullcontext)():
        f = enc(xb.to(device), (depth,), widths=widths)[depth].float().mean(1)
    return [f[j, : widths[j]] for j in range(len(xs))]


def head_logits(
    head: Head, feats: Sequence[torch.Tensor], device: torch.device | str = "cpu"
) -> tuple[torch.Tensor, torch.Tensor]:
    """Lot de caractéristiques → (logits [b, T·up, n_out] en float32, trames
    valides de chaque ligne). Sert à l'apprentissage (perte CTC) et à
    :func:`head_logp`."""
    x = nn.utils.rnn.pad_sequence([f.float() for f in feats], batch_first=True)
    lengths = torch.tensor([f.shape[0] for f in feats])
    return head(x.to(device), lengths).float(), lengths * head.up


def head_logp(
    head: Head, feats: Sequence[torch.Tensor], device: torch.device | str = "cpu"
) -> list[torch.Tensor]:
    """Log-probabilités [w_j·up, n_out] de chaque ligne, seules ses trames."""
    logits, frames = head_logits(head, feats, device)
    lp = logits.log_softmax(-1)
    return [lp[j, : int(frames[j])] for j in range(len(feats))]


def ctc_loss(
    head: Head,
    feats: Sequence[torch.Tensor],
    targets: torch.Tensor,
    target_lengths: torch.Tensor,
    loss: nn.CTCLoss,
    device: torch.device | str = "cpu",
) -> torch.Tensor:
    """Perte CTC d'un lot, sur les seules trames valides de chaque ligne."""
    logits, frames = head_logits(head, feats, device)
    lp = logits.log_softmax(-1).transpose(0, 1)
    value: torch.Tensor = loss(lp, targets, frames, target_lengths)
    return value


def read_lines(
    enc: nn.Module,
    head: Head,
    xs: Sequence[torch.Tensor],
    depth: int,
    chars: Sequence[str],
    white: torch.Tensor,
    device: torch.device | str = "cpu",
    amp: Callable[[], AbstractContextManager[object]] | None = None,
    patch: int = 14,
) -> list[tuple[str, torch.Tensor]]:
    """Lecture d'un lot d'images : (texte glouton, log-probabilités) par
    ligne, dans l'ordre de ``xs``. Indépendante du lot."""
    feats = encode_lines(enc, xs, depth, white, device, amp, patch)
    return [(greedy(lp, chars), lp) for lp in head_logp(head, feats, device)]

# --- axel/reader/metric.py, recopié par le générateur : ne pas modifier ici ---
import re
import unicodedata
from collections.abc import Callable, Iterable

from rapidfuzz.distance import Levenshtein

#: Pliages du scorer HIPE (D-11) ; ``ſ`` est replié avant, par :func:`d150`.
HIPE_TABLE = {
    "_": " ",
    "aͤ": "ä",
    "oͤ": "ö",
    "uͤ": "ü",
    "ß": "ss",
    "ꝛ": "r",
    "œ": "oe",
    "æ": "ae",
}


def normalize_hipe(text: str) -> str:
    """La normalisation du scorer HIPE : minuscules, pliages ß/ꝛ/œ/æ,
    tout non-mot devient une espace, blancs compactés. ``ſ`` est gardé."""
    text = text.lower()
    for src, dst in HIPE_TABLE.items():
        text = text.replace(src, dst)
    text = re.sub(r"[^\w]", " ", text)
    return " ".join(text.split())


def d150(text: str) -> str:
    """Texte normalisé selon D-150 : NFC, puis HIPE avec ``ſ`` replié (sans
    NFC, HIPE change un accent décomposé en espace : D-422)."""
    return normalize_hipe(unicodedata.normalize("NFC", text).replace("ſ", "s"))


def raw(text: str) -> str:
    """Texte brut de l'annexe : NFC, blancs compactés."""
    return " ".join(unicodedata.normalize("NFC", text).split())


NORMS: dict[str, Callable[[str], str]] = {"d150": d150, "raw": raw}


def lev(a: str, b: str) -> int:
    return int(Levenshtein.distance(a, b))


def line_errors(hyp: str | None, truth: str, mode: str = "d150") -> tuple[int, int]:
    """(erreurs, support) d'une ligne ; ``(0, 0)`` si la vérité normalisée
    est vide (la ligne sort du CER, voir :func:`score`)."""
    f = NORMS[mode]
    h, t = f(hyp or ""), f(truth)
    if not t:
        return 0, 0
    return lev(h, t), len(t)


def cer(pairs: Iterable[tuple[str | None, str]], mode: str = "d150") -> float:
    """CER micro en %, ``pairs`` = [(lecture, vérité)]."""
    e = n = 0
    for h, t in pairs:
        a, b = line_errors(h, t, mode)
        e += a
        n += b
    return 100 * e / n if n else 0.0


def score(pairs: Iterable[tuple[str | None, str]]) -> dict[str, float | int]:
    """D-150 d'abord, brut en annexe, et les lignes à vérité vide à part."""
    ps = list(pairs)
    out: dict[str, float | int] = {
        "cer_d150": cer(ps, "d150"),
        "cer_raw": cer(ps, "raw"),
        "lines": len(ps),
    }
    for mode in ("d150", "raw"):
        f = NORMS[mode]
        empty = [(h, t) for h, t in ps if not f(t)]
        out[f"empty_truth_lines_{mode}"] = len(empty)
        out[f"empty_truth_output_chars_{mode}"] = sum(len(f(h or "")) for h, _ in empty)
    return out

# --- axel/reader/ckpt.py, recopié par le générateur : ne pas modifier ici ---
import os
from typing import Any

import torch


def atomic_save(obj: Any, path: str) -> None:
    """``torch.save`` sans jamais laisser ``path`` partiel."""
    tmp = f"{path}.tmp{os.getpid()}"
    try:
        torch.save(obj, tmp)
        torch.load(tmp, map_location="cpu", weights_only=False)  # relecture
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def save_best(path: str, modules: dict[str, Any], score: float, epoch: int) -> None:
    """Meilleur point : états des modules + score (D-150) et époque."""
    atomic_save({"modules": modules, "meta": {"score": score, "epoch": epoch}}, path)


def load_best(
    path: str, expect_score: float | None = None, tol: float = 1e-9
) -> tuple[dict[str, Any], dict[str, Any]]:
    """(états des modules, méta) ; refuse un point dont le score n'est pas
    celui attendu (journal ou point de contrôle de la phase)."""
    d = torch.load(path, map_location="cpu", weights_only=False)
    if "modules" not in d or "meta" not in d:
        raise ValueError(f"{path} : pas un meilleur point C3 (modules + meta)")
    if expect_score is not None and abs(float(d["meta"]["score"]) - expect_score) > tol:
        raise ValueError(
            f"{path} : score {d['meta']['score']} au lieu de {expect_score} attendu"
        )
    return d["modules"], d["meta"]

In [ ]:
DEV_T = 'cuda'
CAP = torch.cuda.get_device_capability()
DT = torch.bfloat16 if CAP[0] >= 8 else torch.float16    # T4 (7.5) : fp16 ; L4/A100 : bf16
HEIGHT, MAXW, UP = 56, 2800, 2
assert (REPO, REV) == (SPECS['dots_mocr']['repo'], SPECS['dots_mocr']['rev'])

def load_encoder(blocks, dtype=None):
    # D-269a : type lu à l'appel (DT peut passer en float32 après S), pas figé à la définition
    dtype = dtype or DT
    s = SPECS['dots_mocr']
    m = NaViT(s['dim'], s['heads'], s['ffn'], blocks, s['rope'])
    sd = {}
    with safe_open(VIS, framework='pt') as fh:
        for k, v in rename_map(s['prefix'], blocks).items():
            sd[v] = fh.get_tensor(k)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    assert not unexpected and [k for k in missing if k != 'inv_freq'] == []
    return m.to(DEV_T, dtype).eval()

clean = raw   # vérité du codec (NFC, blancs compactés)
chars = sorted({c for it in TRAIN for c in clean(it['truth'])})
IDX = {c: i + 1 for i, c in enumerate(chars)}
print(len(chars), 'caractères', DT)
WHITE = white_value(MEAN, STD)

def amp():
    return torch.autocast('cuda', dtype=DT)

def line_images(its):
    return [prep(CROPS[it['stem']], HEIGHT, MAXW) for it in its]

def bucketed(its, bs, shuffle=True):
    # lots de largeurs voisines : moins de blanc calculé pour rien (le résultat n'en dépend plus, D-420)
    key = lambda it: CROPS[it['stem']].width * HEIGHT / max(1, CROPS[it['stem']].height)
    s = sorted(its, key=key)
    bs_ = [s[i:i + bs] for i in range(0, len(s), bs)]
    if shuffle:
        random.shuffle(bs_)
    return bs_

def feats_at(enc, its, depth):
    # D-420 : encodeur masqué ; [w_j, 1536] par ligne, indépendant du lot
    return encode_lines(enc, line_images(its), depth, WHITE, DEV_T, amp)

def store(f):
    # D-269a : fp16 si fini, sinon bf16 (mêmes bornes que float32) ; la tête relit en float32
    h = f.half()
    return (h if torch.isfinite(h).all() else f.bfloat16()).cpu()

def targets(its):
    ys = [torch.tensor([IDX[c] for c in clean(it['truth']) if c in IDX]) for it in its]
    return torch.cat(ys), torch.tensor([len(y) for y in ys])

def read_dev(enc, head, its, depth, dump=None, bs=32):
    # la lecture du choix et de l'export (D-421) : (score D-150 d'abord, lignes)
    head.eval(); pairs, rows = [], []
    with torch.no_grad():
        for b in bucketed(its, bs, shuffle=False):
            for it, lp in zip(b, head_logp(head, feats_at(enc, b, depth), DEV_T)):
                h = greedy(lp, chars); pairs.append((h, it['truth']))
                if dump is not None:
                    rows.append({'stem': it['stem'], 'truth': clean(it['truth']), 'hyp': h,
                                 'logp': lp.half().cpu().numpy().tolist() if dump == 'full' else None})
    head.train()
    return score(pairs), rows

In [ ]:
# --- reader/lecteur/ancre.py, recopié par le générateur : ne pas modifier ici ---
"""Ancre — lecteur ancré : encodeur gelé + tête CTC + décodeur d'attention léger,
décodage joint par score de préfixe CTC (Watanabe et al. 2017 ; Wick et al.
2021 pour l'écriture manuscrite) avec un « couloir » : un préfixe dont le
score CTC tombe à plus de ``corridor`` nats sous le meilleur préfixe CTC de
même longueur est coupé. Le décodeur de langue ne choisit donc qu'entre
lectures que l'image soutient. La géométrie vient de l'alignement forcé du
texte final sur le treillis CTC (positions par caractère).

Entrée : features [B, T, D] d'un encodeur gelé (jetons pré-fusion, une
colonne par 14 px, E17) et longueurs valides. Aucune dépendance à axel.
"""


import math
from dataclasses import dataclass, field

import numpy as np
import torch
from torch import nn

BLANK = 0  # classe blanche du CTC ; le décodeur utilise les mêmes indices
NEG = -1e30


class CTCHead(nn.Module):
    """Tête d'E17 (même architecture que ``axel.reader.ctc.Head``)."""

    def __init__(self, n_out: int, din: int = 1536, hid: int = 256, up: int = 2) -> None:
        super().__init__()
        self.up = up
        self.inp = nn.Sequential(nn.LayerNorm(din), nn.Dropout(0.1), nn.Linear(din, hid))
        self.rnn = nn.LSTM(hid, hid, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1)
        self.out = nn.Linear(2 * hid, n_out)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """-> (log-probas [B, T·up, C], mémoire [B, T·up, 2·hid] pour le décodeur)."""
        z = self.inp(x).repeat_interleave(self.up, dim=1)
        n = (lengths.cpu() * self.up).long()
        packed = nn.utils.rnn.pack_padded_sequence(z, n, batch_first=True, enforce_sorted=False)
        y = nn.utils.rnn.pad_packed_sequence(self.rnn(packed)[0], batch_first=True, total_length=z.shape[1])[0]
        return self.out(y).log_softmax(-1), y


class AttnDecoder(nn.Module):
    """Décodeur autorégressif léger, attention croisée sur la mémoire de la
    tête CTC. Vocabulaire = codec CTC ; l'indice 0 (blanc) sert de BOS/EOS."""

    def __init__(self, n_out: int, dmem: int = 512, d: int = 256, layers: int = 3, heads: int = 4,
                 max_len: int = 512) -> None:
        super().__init__()
        self.emb = nn.Embedding(n_out, d)
        self.pos = nn.Embedding(max_len, d)
        self.mem = nn.Linear(dmem, d)
        layer = nn.TransformerDecoderLayer(d, heads, 4 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.dec = nn.TransformerDecoder(layer, layers)
        self.out = nn.Linear(d, n_out)

    def forward(self, prev: torch.Tensor, memory: torch.Tensor, mem_pad: torch.Tensor) -> torch.Tensor:
        """prev [B, U] (BOS en tête) -> log-probas [B, U, C] du jeton suivant."""
        u = prev.shape[1]
        h = self.emb(prev) + self.pos(torch.arange(u, device=prev.device))[None]
        causal = torch.triu(torch.ones(u, u, dtype=torch.bool, device=prev.device), 1)
        y = self.dec(h, self.mem(memory), tgt_mask=causal, memory_key_padding_mask=mem_pad)
        return self.out(y).log_softmax(-1)


class Ancre(nn.Module):
    def __init__(self, n_out: int, din: int = 1536, hid: int = 256, dec_layers: int = 3) -> None:
        super().__init__()
        self.ctc = CTCHead(n_out, din, hid)
        self.dec = AttnDecoder(n_out, 2 * hid, layers=dec_layers)

    def losses(self, x: torch.Tensor, lengths: torch.Tensor, targets: list[list[int]]) -> dict[str, torch.Tensor]:
        dev = x.device
        logp, mem = self.ctc(x, lengths)
        tl = lengths.to(dev) * self.ctc.up
        flat = torch.tensor([c for t in targets for c in t], dtype=torch.long, device=dev)
        tlen = torch.tensor([len(t) for t in targets], dtype=torch.long, device=dev)
        ctc = nn.functional.ctc_loss(logp.transpose(0, 1).float(), flat, tl, tlen, blank=BLANK,
                                     reduction="mean", zero_infinity=True)
        u = max(len(t) for t in targets) + 1
        prev = torch.zeros(len(targets), u, dtype=torch.long)
        gold = torch.full((len(targets), u), -100, dtype=torch.long)
        for i, t in enumerate(targets):
            prev[i, 1 : len(t) + 1] = torch.tensor(t, dtype=torch.long)
            gold[i, : len(t)] = torch.tensor(t, dtype=torch.long)
            gold[i, len(t)] = BLANK  # EOS
        prev, gold = prev.to(dev), gold.to(dev)
        pad = torch.arange(logp.shape[1], device=dev)[None] >= tl[:, None]
        att = nn.functional.nll_loss(self.dec(prev, mem, pad).transpose(1, 2), gold, ignore_index=-100)
        return {"ctc": ctc, "att": att}


# -- score de préfixe CTC ------------------------------------------------------


@dataclass
class CTCPrefixState:
    """Variables avant r^n_t(g), r^b_t(g) du préfixe g (log), sur tous les t."""

    rn: np.ndarray
    rb: np.ndarray
    last: int
    score: float  # log P(g... | x) : préfixe g suivi de n'importe quoi


def ctc_prefix_init(lp: np.ndarray) -> CTCPrefixState:
    t = lp.shape[0]
    rn = np.full(t, NEG)
    rb = np.cumsum(lp[:, BLANK])
    return CTCPrefixState(rn, rb, -1, 0.0)


def ctc_prefix_extend(lp: np.ndarray, st: CTCPrefixState, c: int) -> CTCPrefixState:
    """Score de préfixe de g+c (Watanabe 2017, algorithme 2), O(T)."""
    T = lp.shape[0]
    rn = np.full(T, NEG)
    rb = np.full(T, NEG)
    prev_tot = np.logaddexp(st.rn, st.rb)
    # phi_t = r_t(g) sauf si c répète le dernier caractère : alors seul r^b
    phi = st.rb.copy() if c == st.last else prev_tot
    start = st.last < 0
    rn[0] = lp[0, c] if start else NEG
    psi = rn[0]
    for t in range(1, T):
        rn[t] = np.logaddexp(rn[t - 1], phi[t - 1]) + lp[t, c]
        rb[t] = np.logaddexp(rn[t - 1], rb[t - 1]) + lp[t, BLANK]
        psi = np.logaddexp(psi, phi[t - 1] + lp[t, c])
    return CTCPrefixState(rn, rb, c, float(psi))


def ctc_prefix_extend_many(lp: np.ndarray, sts: list[CTCPrefixState], cs: list[int]) -> list[CTCPrefixState]:
    """:func:`ctc_prefix_extend` pour N couples (état, caractère) à la fois :
    même récurrence, vectorisée sur les candidats (boucle sur T seulement)."""
    if not sts:
        return []
    T = lp.shape[0]
    n = len(sts)
    c = np.asarray(cs)
    last = np.asarray([s.last for s in sts])
    prn = np.stack([s.rn for s in sts])  # [N, T]
    prb = np.stack([s.rb for s in sts])
    phi = np.where((c == last)[:, None], prb, np.logaddexp(prn, prb))
    xc = lp[:, c].T  # [N, T]
    xb = lp[:, BLANK]
    rn = np.full((n, T), NEG)
    rb = np.full((n, T), NEG)
    rn[:, 0] = np.where(last < 0, xc[:, 0], NEG)
    psi = rn[:, 0].copy()
    for t in range(1, T):
        rn[:, t] = np.logaddexp(rn[:, t - 1], phi[:, t - 1]) + xc[:, t]
        rb[:, t] = np.logaddexp(rn[:, t - 1], rb[:, t - 1]) + xb[t]
        psi = np.logaddexp(psi, phi[:, t - 1] + xc[:, t])
    return [CTCPrefixState(rn[i], rb[i], int(c[i]), float(psi[i])) for i in range(n)]


def ctc_full(st: CTCPrefixState) -> float:
    """log P(g | x) exact pour le préfixe complet (fin de ligne)."""
    return float(np.logaddexp(st.rn[-1], st.rb[-1]))


# -- décodage joint ancré ------------------------------------------------------


@dataclass
class Hyp:
    ids: tuple[int, ...]
    ctc: CTCPrefixState
    att: float
    score: float = 0.0
    done: bool = False
    final_ctc: float = field(default=NEG)


@torch.no_grad()
def joint_decode(model: Ancre, x: torch.Tensor, length: int, alpha: float = 0.5, beam: int = 8,
                 topk: int = 8, corridor: float = 15.0, max_len: int | None = None,
                 lm=None, lm_weight: float = 0.0) -> list[Hyp]:
    """Une ligne. score = α·CTC + (1−α)·attention (+ λ·LM externe).
    ``corridor`` : coupe un préfixe dont le score CTC est à plus de ``corridor``
    nats sous le meilleur score CTC de préfixe à la même étape."""
    model.eval()
    lengths = torch.tensor([length])
    logp, mem = model.ctc(x[:, :length], lengths)
    lp = logp[0].double().cpu().numpy()
    T = lp.shape[0]
    max_len = max_len or T // 2 + 2
    dev = x.device
    pad = torch.zeros(1, T, dtype=torch.bool, device=dev)
    hyps = [Hyp((), ctc_prefix_init(lp), 0.0)]
    finished: list[Hyp] = []
    for _ in range(max_len):
        prev = torch.tensor([[BLANK] + list(h.ids) for h in hyps], device=dev)
        att = model.dec(prev, mem.expand(len(hyps), -1, -1), pad.expand(len(hyps), -1))[:, -1].double().cpu().numpy()
        cand: list[Hyp] = []
        pend: list[tuple[Hyp, int, float]] = []
        for h, a in zip(hyps, att):
            for c in np.argsort(-a)[:topk]:
                c = int(c)
                if c == BLANK:  # fin
                    fc = ctc_full(h.ctc)
                    s = alpha * fc + (1 - alpha) * (h.att + a[c])
                    if lm is not None:
                        s += lm_weight * lm(h.ids, None)
                    cand.append(Hyp(h.ids, h.ctc, h.att + a[c], s, True, fc))
                else:
                    pend.append((h, c, float(a[c])))
        sts = ctc_prefix_extend_many(lp, [p[0].ctc for p in pend], [p[1] for p in pend])
        for (h, c, ac), st in zip(pend, sts):
            s = alpha * st.score + (1 - alpha) * (h.att + ac)
            if lm is not None:
                s += lm_weight * lm(h.ids, c)
            cand.append(Hyp(h.ids + (c,), st, h.att + ac, s))
        live = [h for h in cand if not h.done]
        if live:
            best_ctc = max(h.ctc.score for h in live)
            live = [h for h in live if h.ctc.score >= best_ctc - corridor]
        finished += [h for h in cand if h.done]
        live.sort(key=lambda h: -h.score)
        hyps = live[:beam]
        if not hyps or (finished and max(f.score for f in finished) > hyps[0].score):
            break
    return sorted(finished, key=lambda h: -h.score) or sorted(hyps, key=lambda h: -h.score)


# -- géométrie : alignement forcé ------------------------------------------------


def forced_align(lp: np.ndarray, ids: list[int]) -> list[tuple[int, int, float]]:
    """Viterbi CTC : pour chaque caractère, (premier pas, dernier pas, log-proba
    moyenne de ses pas). ``[]`` si le texte est infaisable pour T."""
    T = lp.shape[0]
    ext = [BLANK]
    for c in ids:
        ext += [c, BLANK]
    S = len(ext)
    if not ids:
        return []
    dp = np.full((T, S), NEG)
    bp = np.zeros((T, S), dtype=np.int64)
    dp[0, 0] = lp[0, BLANK]
    dp[0, 1] = lp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            best, arg = dp[t - 1, s], s
            if s >= 1 and dp[t - 1, s - 1] > best:
                best, arg = dp[t - 1, s - 1], s - 1
            if s >= 2 and ext[s] != BLANK and ext[s] != ext[s - 2] and dp[t - 1, s - 2] > best:
                best, arg = dp[t - 1, s - 2], s - 2
            dp[t, s] = best + lp[t, ext[s]]
            bp[t, s] = arg
    s = S - 1 if dp[T - 1, S - 1] >= dp[T - 1, S - 2] else S - 2
    if dp[T - 1, s] <= NEG / 2:
        return []
    path = [s]
    for t in range(T - 1, 0, -1):
        s = bp[t, s]
        path.append(s)
    path.reverse()
    out: list[list] = [[None, None, []] for _ in ids]
    for t, s in enumerate(path):
        if s % 2 == 1:
            k = s // 2
            if out[k][0] is None:
                out[k][0] = t
            out[k][1] = t
            out[k][2].append(lp[t, ext[s]])
    return [(a, b, float(np.mean(v))) for a, b, v in out]


def readability_margin(lp: np.ndarray) -> float:
    """Marge moyenne (top1 − top2, en nats) sur les pas non blancs : une
    image illisible donne un treillis plat (E16, E18)."""
    srt = np.sort(lp, axis=1)
    keep = lp.argmax(1) != BLANK
    if not keep.any():
        return math.inf
    return float((srt[keep, -1] - srt[keep, -2]).mean())


# -- faisceau joint sur plusieurs treillis CTC (encodeurs différents) -------------


@dataclass
class MultiHyp:
    text: str
    states: list[CTCPrefixState]
    lm: float
    score: float = 0.0
    done: bool = False


def _extend_ids(lp: np.ndarray, sts: list[CTCPrefixState], seqs: list[list[int]]) -> list[CTCPrefixState]:
    """Étend chaque état par une suite d'indices (un caractère du texte peut
    valoir plusieurs classes dans un codec : é = e + accent en NFD)."""
    out = list(sts)
    depth = max((len(s) for s in seqs), default=0)
    for d in range(depth):
        idx = [i for i, s in enumerate(seqs) if len(s) > d]
        new = ctc_prefix_extend_many(lp, [out[i] for i in idx], [seqs[i][d] for i in idx])
        for i, st in zip(idx, new):
            out[i] = st
    return out


def multi_ctc_beam(lattices: list[tuple[np.ndarray, float, "callable"]], candidates: list[str],
                   lm=None, lm_weight: float = 0.0, beta: float = 0.0, beam: int = 8,
                   corridor: float = 20.0, max_len: int = 400) -> list[MultiHyp]:
    """Faisceau synchrone sur les caractères du texte. ``lattices`` : (log-probas
    [T_i, C_i], poids w_i, encode_i(car) -> liste d'indices ou None). Score d'un
    préfixe g : Σ w_i·log P_i(g…) + λ·log P_LM(g) + β·|g| ; fin : Σ w_i·log P_i(g).
    Un caractère qu'un codec ne sait pas écrire rend ce préfixe impossible pour
    lui (−∞) : rien n'est écrit qu'aucun treillis n'appuie. ``corridor`` : coupe
    un préfixe dont la somme pondérée CTC est à plus de ``corridor`` nats sous la
    meilleure de l'étape (le LM ne peut pas tirer loin de l'image)."""
    root = MultiHyp("", [ctc_prefix_init(lp) for lp, _, _ in lattices], 0.0)
    hyps, finished = [root], []
    for _ in range(max_len):
        # fins
        for h in hyps:
            full = sum(w * ctc_full(st) for st, (_, w, _) in zip(h.states, lattices))
            lmv = h.lm + (lm(h.text, None) if lm is not None else 0.0)
            finished.append(MultiHyp(h.text, h.states, lmv, full + lm_weight * lmv + beta * len(h.text), True))
        pairs = [(h, c) for h in hyps for c in candidates]
        new_states: list[list[CTCPrefixState | None]] = [[None] * len(lattices) for _ in pairs]
        for li, (lp, _, enc) in enumerate(lattices):
            ok = [(k, enc(c)) for k, (_, c) in enumerate(pairs)]
            ok = [(k, s) for k, s in ok if s]
            ext = _extend_ids(lp, [pairs[k][0].states[li] for k, _ in ok], [s for _, s in ok])
            for (k, _), st in zip(ok, ext):
                new_states[k][li] = st
        live = []
        for (h, c), sts in zip(pairs, new_states):
            if any(s is None for s in sts):
                continue
            vis = sum(w * st.score for st, (_, w, _) in zip(sts, lattices))
            lmv = h.lm + (lm(h.text, c) if lm is not None else 0.0)
            live.append((vis, MultiHyp(h.text + c, sts, lmv, vis + lm_weight * lmv + beta * (len(h.text) + 1))))
        if not live:
            break
        best_vis = max(v for v, _ in live)
        live = [h for v, h in live if v >= best_vis - corridor]
        live.sort(key=lambda h: -h.score)
        hyps = live[:beam]
        if finished and max(f.score for f in finished) >= hyps[0].score + 5.0 * max(1.0, beta):
            break
    return sorted(finished, key=lambda h: -h.score)


In [ ]:
# F — caractéristiques L = 32, une fois, sur le disque local (fp16, ~18 Go)
FEAT = f'{TMP}/feats'; os.makedirs(FEAT, exist_ok=True)
INDEX = {}
ITEMS_DEV = list(DEV)
if SMOKE:
    TRAIN = random.Random(0).sample(TRAIN, 2000); DEV = random.Random(0).sample(DEV, 300)
if 'F' in PHASES and not os.path.exists(f'{FEAT}/index.json'):
    enc = load_encoder(L)
    t0 = time.time(); k = 0
    for split, its in (('train', TRAIN), ('dev', DEV)):
        for b in bucketed(its, 32, shuffle=False):
            with torch.no_grad():
                fs = feats_at(enc, b, L)
            torch.save([store(f) for f in fs], f'{FEAT}/{k}.pt')
            for j, it in enumerate(b):
                INDEX[it['stem']] = (k, j)
            k += 1
            if k % 200 == 0:
                print(split, k, round(time.time() - t0), 's', flush=True)
    json.dump(INDEX, open(f'{FEAT}/index.json', 'w'))
    del enc; torch.cuda.empty_cache()
    print('caractéristiques', len(INDEX), round(time.time() - t0), 's')
INDEX = json.load(open(f'{FEAT}/index.json'))
SHARDS = {}
for stem, (k, j) in INDEX.items():
    SHARDS.setdefault(k, []).append(stem)
STEM2IT = {it['stem']: it for it in TRAIN + DEV}
DEV_STEMS = {it['stem'] for it in DEV}
# Partition tune/eval par page (graine 1, sha256 eval 996ab0cd… dans READER_R3_PREREG) :
# fixée AVANT l'apprentissage ; le choix d'époque ne voit que tune (revue pM, 15/10).
def page_of(stem):
    return stem.split('__')[1] if '__' in stem else stem
ALL_DEV = sorted(it['stem'] for it in ITEMS_DEV)
pages = sorted({page_of(s) for s in ALL_DEV}); random.Random(1).shuffle(pages)
TUNE_P = set(pages[: len(pages) // 2])
TUNE_STEMS = {s for s in DEV_STEMS if page_of(s) in TUNE_P}
print('dev', len(DEV_STEMS), 'dont tune', len(TUNE_STEMS))

In [ ]:
# T — Ancre : tête CTC partie de head_a1.pt, décodeur neuf ; perte λ·CTC + (1−λ)·attention
_c = sorted(glob.glob(f'{D}/**/chars.json', recursive=True))
_tj = [p for p in sorted(glob.glob(f'{D}/**/timing.json', recursive=True)) if 'chars' in json.load(open(p))]
if _c:
    assert len(_c) == 1, f'plusieurs chars.json : {_c}'
    A1_CHARS = json.load(open(_c[0]))
else:
    assert len(_tj) == 1, f'il faut exactement un timing.json avec chars : {_tj}'
    A1_CHARS = json.load(open(_tj[0]))['chars']
assert len(A1_CHARS) == 141, len(A1_CHARS)
_h = sorted(glob.glob(f'{D}/**/head_a1.pt', recursive=True))
assert len(_h) == 1 and _h[0].endswith('head_a1.pt'), f'il faut exactement un head_a1.pt : {_h}'
sd = torch.load(_h[0], map_location='cpu', weights_only=False)
assert isinstance(sd, dict) and 'out.weight' in sd, 'head_a1.pt doit être le state_dict de la tête A1'
print('tête A1 :', _h[0], '| codec :', _c[0] if _c else _tj[0])
assert A1_CHARS == chars, "codec différent de celui d'A1 (ordre compris)"
model = Ancre(len(chars) + 1).to(DEV_T)
print(model.ctc.load_state_dict(sd, strict=True))
opt = torch.optim.AdamW([{'params': model.ctc.parameters(), 'lr': 2e-4},
                         {'params': model.dec.parameters(), 'lr': 1e-3}], weight_decay=0.01)

def batches(stems_by_shard, train):
    ks = list(stems_by_shard)
    if train:
        random.shuffle(ks)
    for k in ks:
        fs = torch.load(f'{FEAT}/{k}.pt')
        stems = SHARDS[k]
        keep = [(f, s) for f, s in zip(fs, stems) if s in stems_by_shard[k]]
        if keep:
            yield keep

def collate(keep):
    fs = [f.float() for f, _ in keep]
    n = torch.tensor([f.shape[0] for f in fs])
    x = torch.zeros(len(fs), int(n.max()), fs[0].shape[1])
    for i, f in enumerate(fs):
        x[i, : f.shape[0]] = f
    ys = [[IDX[c] for c in clean(STEM2IT[s]['truth']) if c in IDX] for _, s in keep]
    return x.to(DEV_T), n.to(DEV_T), ys, [s for _, s in keep]

TR_SH = {k: set(v) - DEV_STEMS for k, v in SHARDS.items()}
TR_SH = {k: v for k, v in TR_SH.items() if v}
DV_SH = {k: set(v) & DEV_STEMS for k, v in SHARDS.items()}
DV_SH = {k: v for k, v in DV_SH.items() if v}
TUNE_SH = {k: v & TUNE_STEMS for k, v in DV_SH.items() if v & TUNE_STEMS}

@torch.no_grad()
def greedy_both(x, n):
    # CTC glouton et attention gloutonne (en lot), même modèle
    model.eval()
    logp, mem = model.ctc(x, n)
    tl = n * model.ctc.up
    pad = torch.arange(logp.shape[1], device=x.device)[None] >= tl[:, None]
    B = x.shape[0]; prev = torch.zeros(B, 1, dtype=torch.long, device=x.device)
    done = torch.zeros(B, dtype=torch.bool, device=x.device)
    for _ in range(int(tl.max()) // 2 + 2):
        nxt = model.dec(prev, mem, pad)[:, -1].argmax(-1)
        nxt = torch.where(done, torch.zeros_like(nxt), nxt)
        prev = torch.cat([prev, nxt[:, None]], 1); done |= nxt == 0
        if done.all():
            break
    model.train()
    att = []
    for row in prev[:, 1:].tolist():
        s = []
        for c in row:
            if c == 0:
                break
            s.append(chars[c - 1])
        att.append(''.join(s))
    ctc = [greedy(lp[: int(t)].float().cpu(), chars) for lp, t in zip(logp, tl)]
    return ctc, att, logp, tl

def eval_dev():
    # choix d'époque : moitié TUNE seulement (aucune ligne eval)
    pc, pa = [], []
    for keep in batches(TUNE_SH, False):
        x, n, ys, stems = collate(keep)
        c, a, _, _ = greedy_both(x, n)
        for s, hc, ha in zip(stems, c, a):
            t = STEM2IT[s]['truth']; pc.append((hc, t)); pa.append((ha, t))
    return score(pc), score(pa)

LOG = open(f'{OUT}/r3_train_log.jsonl', 'a')
if 'T' in PHASES:
    sc0 = eval_dev()
    print('départ (A1 + décodeur neuf) : CTC', sc0[0]['cer_d150'], 'attention', sc0[1]['cer_d150'], flush=True)
    best = 1e9
    for ep in range(1, EPOCHS + 1):
        if time.time() - T_START > BUDGET_H * 3600:
            break
        t0 = time.time(); ls = []
        for keep in batches(TR_SH, True):
            x, n, ys, _ = collate(keep)
            ok = [i for i, y in enumerate(ys) if y]
            if not ok:
                continue
            x, n, ys = x[ok], n[ok], [ys[i] for i in ok]
            with torch.autocast('cuda', dtype=DT):
                lo = model.losses(x, n.cpu(), ys)
            loss = LAMBDA_CTC * lo['ctc'] + (1 - LAMBDA_CTC) * lo['att']
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            ls.append((lo['ctc'].item(), lo['att'].item()))
        sc = eval_dev()
        row = {'epoch': ep, 'selection': 'tune', 'ctc_loss': float(np.mean([a for a, _ in ls])), 'att_loss': float(np.mean([b for _, b in ls])),
               'dev_ctc_d150': sc[0]['cer_d150'], 'dev_att_d150': sc[1]['cer_d150'], 'epoch_s': time.time() - t0}
        LOG.write(json.dumps(row) + '\n'); LOG.flush(); print(row, flush=True)
        crit = min(sc[0]['cer_d150'], sc[1]['cer_d150'])
        if crit < best - 0.01:
            best = crit; torch.save(model.state_dict(), f'{OUT}/r3_best.pt.tmp'); os.replace(f'{OUT}/r3_best.pt.tmp', f'{OUT}/r3_best.pt')
model.load_state_dict(torch.load(f'{OUT}/r3_best.pt', map_location=DEV_T))

In [ ]:
# E — décodage joint ancré : réglage (α, couloir) sur 300 lignes de la moitié tune, mesure sur la moitié eval
tune = sorted(TUNE_STEMS)
evl = sorted(s for s in DEV_STEMS if page_of(s) not in TUNE_P)
if SMOKE:
    # fumée : aucune ligne de la moitié eval n'est décodée ni imprimée (revue pM, 15/10)
    evl = tune[: min(50, len(tune))]
FEATS_OF = {}
for keep in batches(DV_SH, False):
    for f, s in keep:
        FEATS_OF[s] = f

def ids2txt(ids):
    return ''.join(chars[c - 1] for c in ids)

def run_joint(stems, alpha, corridor, beam=8, topk=8):
    out = {}
    for s in stems:
        f = FEATS_OF[s].float()[None].to(DEV_T)
        hs = joint_decode(model, f, f.shape[1], alpha=alpha, beam=beam, topk=topk, corridor=corridor)
        out[s] = ids2txt(hs[0].ids) if hs else ''
    return out

RES = {'tune_lines': len(tune), 'eval_lines': len(evl)}
if 'E' in PHASES:
    sub = random.Random(3).sample(tune, min(30 if SMOKE else 300, len(tune)))
    grid = {}
    for alpha in (0.3, 0.5, 0.7, 1.0):
        for corridor in (10.0, 30.0):
            t0 = time.time()
            o = run_joint(sub, alpha, corridor)
            grid[f'{alpha}/{corridor}'] = score([(o[s], STEM2IT[s]['truth']) for s in sub])['cer_d150']
            print('tune', alpha, corridor, grid[f'{alpha}/{corridor}'], round(time.time() - t0), 's', flush=True)
    RES['grid_tune300'] = grid
    # égalité : plus grand α (plus de poids au CTC), puis plus petit couloir (règle fixée avant mesure)
    a, c = map(float, min(grid, key=lambda k: (grid[k], -float(k.split('/')[0]), float(k.split('/')[1]))).split('/'))
    RES['chosen'] = {'alpha': a, 'corridor': c}
    oj = run_joint(evl, a, c)
    pc, pa, rows = [], [], []
    for keep in batches({k: v & set(evl) for k, v in DV_SH.items() if v & set(evl)}, False):
        x, n, ys, stems = collate(keep)
        cg, ag, logp, tl = greedy_both(x, n)
        for i, s in enumerate(stems):
            t = STEM2IT[s]['truth']; pc.append((cg[i], t)); pa.append((ag[i], t))
            lp = logp[i, : int(tl[i])].float().cpu().numpy()
            rows.append({'stem': s, 'truth': t, 'ctc': cg[i], 'att': ag[i], 'joint': oj[s],
                         'margin': readability_margin(lp), 'logp': lp.astype(np.float16).tolist()})
    RES['eval_ctc_greedy'] = score(pc); RES['eval_att_greedy'] = score(pa)
    RES['eval_joint'] = score([(oj[s], STEM2IT[s]['truth']) for s in evl])
    with gzip.open(f'{OUT}/r3_dev_eval.jsonl.gz', 'wt') as g:
        for r in rows:
            g.write(json.dumps(r, ensure_ascii=False) + '\n')
    json.dump(RES, open(f'{OUT}/r3_results.json', 'w'), indent=1)
    print({k: (v['cer_d150'] if isinstance(v, dict) and 'cer_d150' in v else v) for k, v in RES.items()})

In [ ]:
# I — invention : 200 lignes de la moitié eval × {blanc, brouillé 28 px, flou fort, bruit fort}
from PIL import ImageFilter
def perturb(img, mode, key):
    seed = int(hashlib.sha1(key.encode()).hexdigest()[:8], 16)
    if mode == 'blank':
        return Image.new('L', img.size, 255)
    if mode == 'blur_strong':
        return img.filter(ImageFilter.GaussianBlur(max(2.0, img.height / 12)))
    arr = np.asarray(img).copy()
    if mode == 'noise_strong':
        rng = np.random.default_rng(seed)
        return Image.fromarray(np.clip(arr.astype(np.float32) + rng.normal(0, 70, arr.shape), 0, 255).astype(np.uint8))
    h, w = arr.shape[0] // 28, arr.shape[1] // 28
    tiles = [arr[r*28:(r+1)*28, c*28:(c+1)*28].copy() for r in range(h) for c in range(w)]
    random.Random(seed).shuffle(tiles)
    for k, t in enumerate(tiles):
        r, c = divmod(k, w); arr[r*28:(r+1)*28, c*28:(c+1)*28] = t
    return Image.fromarray(arr)

def words3(text):
    ws = [w for w in re.findall(r"[^\W\d_]{2,}", text)]
    return len(ws) >= 3

import re
if 'I' in PHASES:
    enc = load_encoder(L)
    picks = random.Random(4).sample(evl, min(10 if SMOKE else 200, len(evl)))
    a, c = RES.get('chosen', {'alpha': 0.5, 'corridor': 10.0}).values()
    inv = {}
    for mode in ('blank', 'scramble', 'blur_strong', 'noise_strong'):
        rows = []
        for k in range(0, len(picks), 16):
            b = [STEM2IT[s] for s in picks[k:k+16]]
            ims = [prep(perturb(CROPS[it['stem']], mode, it['stem'] + '#r4'), HEIGHT, MAXW) for it in b]
            with torch.no_grad():
                fs = encode_lines(enc, ims, L, WHITE, DEV_T, amp)
            x, n, _, stems = collate([(store(f), it['stem']) for f, it in zip(fs, b)])
            cg, ag, logp, tl = greedy_both(x, n)
            for i, it in enumerate(b):
                f = store(fs[i]).float()[None].to(DEV_T)
                hs = joint_decode(model, f, f.shape[1], alpha=a, corridor=c)
                jt = ids2txt(hs[0].ids) if hs else ''
                lp = logp[i, : int(tl[i])].float().cpu().numpy()
                rows.append({'stem': it['stem'], 'ctc': cg[i], 'att': ag[i], 'joint': jt, 'margin': readability_margin(lp)})
        inv[mode] = {arm: sum(words3(r[arm]) for r in rows) for arm in ('ctc', 'att', 'joint')}
        inv[mode]['lines'] = len(rows)
        inv[mode]['rows'] = rows
        print(mode, {k: v for k, v in inv[mode].items() if k != 'rows'}, flush=True)
    RES['invention'] = {m: {k: v for k, v in d.items() if k != 'rows'} for m, d in inv.items()}
    json.dump(inv, open(f'{OUT}/r3_invention.json', 'w'), ensure_ascii=False)
    json.dump(RES, open(f'{OUT}/r3_results.json', 'w'), indent=1)
RES['hours'] = (time.time() - T_START) / 3600
print(json.dumps({k: v for k, v in RES.items() if k != 'grid_tune300'}, indent=1, ensure_ascii=False))